# 03. ML 학습용 윈도우 데이터셋 생성

이 노트북은 PreDist 운영 시계열을 HeatGrid Agent의 ML 모델이 사용할 수 있는 시간 구간 단위 데이터셋으로 변환한다.

02번까지는 fault, normal, disturbance 라벨이 실제 운영 데이터 시간 범위와 맞는지 확인했다. 03번에서는 그 결과를 사용해서 `기계실 + 시간 구간 + 센서 요약 + 라벨 + 데이터 품질` 형태의 학습 입력 데이터를 만든다.

## 1. 이 단계의 역할

HeatGrid Agent에서 ML은 우선순위를 직접 결정하지 않는다.

ML은 다음 정보를 Agent가 해석할 수 있는 형태로 제공해야 한다.

- 어느 기계실의 구간인지
- 어느 시간 구간인지
- 정상 후보인지, 고장 전 위험 후보인지
- 고장신고까지 몇 시간이 남았는지
- 결측이나 시간 공백 때문에 데이터 신뢰도가 낮은지
- 어떤 센서 변화가 컸는지

따라서 03번의 목표는 모델 학습이 아니라, 모델이 학습할 수 있는 기본 입력 테이블을 만드는 것이다.

In [1]:

from pathlib import Path
import hashlib
import math
import re

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('pyproject.toml? ?? ? ????. ???? ???? ??? ???.')


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw_data' / 'predist_v2'
ALIGNMENT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'label_alignment'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_windows'
OUTPUT_PATH = OUTPUT_DIR / 'ml_window_dataset.csv'
NORMAL_PROFILE_PATH = OUTPUT_DIR / 'normal_profile_by_group.csv'
NORMAL_REFERENCE_DRIFT_PATH = OUTPUT_DIR / 'normal_reference_drift_by_group.csv'
TIME_CONTEXT_PROFILE_PATH = OUTPUT_DIR / 'window_time_context_profile.csv'

WINDOW_SIZE = pd.Timedelta(hours=6)
WINDOW_FREQ = '6h'
MIN_ROW_RATIO = 0.5
MAX_MISSING_RATE = 0.3
KEEP_ONLY_RELEVANT_WINDOWS = True
DISTURBANCE_CONTEXT_HOURS = 6
MAX_PREFAULT_LEAD_HOURS = 168
STABILIZATION_DAYS = 14
HEATING_SEASON_MONTHS = {11, 12, 1, 2, 3}
CONTROL_CONTEXT_PATTERN = re.compile(r'(mode|status|state|operation)', re.IGNORECASE)

CORE_SENSOR_COLUMNS = [
    'outdoor_temperature',
    's_hc1_supply_temperature',
    's_hc1_supply_temperature_setpoint',
    's_dhw_supply_temperature',
    's_dhw_supply_temperature_setpoint',
    'p_hc1_return_temperature',
    'p_dhw_return_temperature',
    's_dhw_upper_storage_temperature',
    's_dhw_lower_storage_temperature',
    'p_net_meter_energy',
    'p_net_meter_volume',
    'p_net_meter_heat_power',
    'p_net_meter_flow',
    'p_net_supply_temperature',
    'p_net_return_temperature',
    'p_hc1_control_valve_position_setpoint',
    'p_dhw_control_valve_position',
]

DERIVED_PAIRS = {
    'hc1_supply_temperature_gap': ('s_hc1_supply_temperature', 's_hc1_supply_temperature_setpoint'),
    'dhw_supply_temperature_gap': ('s_dhw_supply_temperature', 's_dhw_supply_temperature_setpoint'),
    'network_temperature_gap': ('p_net_supply_temperature', 'p_net_return_temperature'),
}

CUMULATIVE_COLUMNS = ['p_net_meter_energy', 'p_net_meter_volume']
TEMPERATURE_COLUMNS = [column for column in CORE_SENSOR_COLUMNS if 'temperature' in column]
FLOW_COLUMNS = [column for column in CORE_SENSOR_COLUMNS if 'flow' in column]
POWER_COLUMNS = [column for column in CORE_SENSOR_COLUMNS if 'heat_power' in column]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'???? ??: {PROJECT_ROOT}')
print(f'??? ??: {WINDOW_SIZE}')
print(f'?? ??: {OUTPUT_PATH}')
print(f'??/?? ?? ??? ??: {KEEP_ONLY_RELEVANT_WINDOWS}')


???? ??: C:\Project3\HeatGrid_Agent
??? ??: 0 days 06:00:00
?? ??: C:\Project3\HeatGrid_Agent\data\processed\ml_windows\ml_window_dataset.csv
??/?? ?? ??? ??: True


## 2. 입력 데이터 불러오기

03번은 02번에서 만든 라벨 정렬 결과를 입력으로 사용한다.

- `fault_alignment.csv`: 고장 전 위험 후보 구간
- `normal_alignment.csv`: 정상 후보 구간
- `disturbance_alignment.csv`: 정비/작업 영향 후보 구간
- `configuration_types.csv`: 기계실 설비 구성

`disturbance`는 고장 라벨로 직접 쓰지 않고, Agent가 해석할 수 있는 보조 정보로 붙인다.

In [2]:
def read_csv(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'필수 파일이 없습니다: {path}')
    return pd.read_csv(path, **kwargs)


def read_predist_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=';', low_memory=False)


def parse_datetime_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    result = df.copy()
    for column in columns:
        if column in result.columns:
            result[column] = pd.to_datetime(result[column], errors='coerce')
    return result


fault_alignment = parse_datetime_columns(
    read_csv(ALIGNMENT_DIR / 'fault_alignment.csv'),
    ['report_date', 'window_start', 'window_end', 'overlap_start', 'overlap_end'],
)
normal_alignment = parse_datetime_columns(
    read_csv(ALIGNMENT_DIR / 'normal_alignment.csv'),
    ['window_start', 'window_end', 'overlap_start', 'overlap_end'],
)
disturbance_alignment = parse_datetime_columns(
    read_csv(ALIGNMENT_DIR / 'disturbance_alignment.csv'),
    ['event_start', 'window_start', 'window_end'],
)
operational_coverage = parse_datetime_columns(
    read_csv(ALIGNMENT_DIR / 'operational_coverage.csv'),
    ['timestamp_min', 'timestamp_max'],
)

fault_alignment = fault_alignment[fault_alignment['is_usable'] == True].copy()
normal_alignment = normal_alignment[normal_alignment['is_usable'] == True].copy()
disturbance_alignment = disturbance_alignment[disturbance_alignment['is_usable'] == True].copy()

input_summary = pd.DataFrame([
    {'입력 파일': 'fault_alignment.csv', '사용 가능 행 수': len(fault_alignment)},
    {'입력 파일': 'normal_alignment.csv', '사용 가능 행 수': len(normal_alignment)},
    {'입력 파일': 'disturbance_alignment.csv', '사용 가능 행 수': len(disturbance_alignment)},
    {'입력 파일': 'operational_coverage.csv', '사용 가능 행 수': len(operational_coverage)},
])
input_summary

,입력 파일,사용 가능 행 수
0,fault_alignment.csv,73
1,normal_alignment.csv,65
2,disturbance_alignment.csv,311
3,operational_coverage.csv,93


In [3]:
def load_configuration_table(raw_root: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(raw_root.glob('manufacturer */configuration_types.csv')):
        manufacturer = path.parent.name
        df = read_predist_csv(path)
        df = df.rename(columns={'substation ID': 'substation_id'})
        df['manufacturer'] = manufacturer
        frames.append(df[['manufacturer', 'substation_id', 'configuration_type']])
    if not frames:
        return pd.DataFrame(columns=['manufacturer', 'substation_id', 'configuration_type', 'has_dhw', 'has_buffer_tank'])

    result = pd.concat(frames, ignore_index=True)
    result['substation_id'] = pd.to_numeric(result['substation_id'], errors='coerce').astype('Int64')
    result['has_dhw'] = result['configuration_type'].str.contains('DHW', case=False, na=False)
    result['has_buffer_tank'] = result['configuration_type'].str.contains('buffer', case=False, na=False)
    return result


configuration_table = load_configuration_table(RAW_ROOT)

configuration_summary = (
    configuration_table
    .groupby(['manufacturer', 'configuration_type'], dropna=False)
    .size()
    .reset_index(name='기계실 수')
    .rename(columns={'manufacturer': '제조사', 'configuration_type': '설비 구성'})
)
configuration_summary

,제조사,설비 구성,기계실 수
0,manufacturer 1,SH + DHW,28
1,manufacturer 1,SH + DHW with sub-circuits,7
2,manufacturer 2,SH,20
3,manufacturer 2,SH + DHW,22
4,manufacturer 2,SH + DHW with sub-circuits,1
5,manufacturer 2,SH with buffer tank,13
6,manufacturer 2,SH with sub-circuits,1
7,manufacturer 2,NaN,1


## 3. 윈도우 생성 함수

초기 기준은 `6시간 윈도우 / 6시간 간격`이다.

겹치는 윈도우를 쓰지 않는 이유는 다음과 같다.

- 비슷한 구간이 train/test에 동시에 들어가는 문제를 줄인다.
- 결과 행을 사람이 확인하기 쉽다.
- baseline 모델을 만들 때 label leakage 위험을 줄인다.

결측치는 feature 계산을 위해 보간하지만, 결측 행 수와 결측률은 별도 feature로 남긴다.
보간은 반드시 같은 윈도우 안에서만 수행한다. 서로 다른 fault, normal, disturbance 구간의 값이 섞이면 안 되기 때문이다.

In [4]:

def extract_substation_id(path: Path) -> int:
    match = re.search(r'substation_(\d+)\.csv$', path.name)
    if not match:
        raise ValueError(f'substation ID? ????? ?? ? ????: {path.name}')
    return int(match.group(1))


def list_operational_files(raw_root: Path) -> list[Path]:
    return sorted(raw_root.glob('manufacturer */operational_data/substation_*.csv'))


def season_bucket_from_month(month: int) -> str:
    if month in {12, 1, 2}:
        return 'winter'
    if month in {3, 4, 5}:
        return 'spring'
    if month in {6, 7, 8}:
        return 'summer'
    return 'autumn'


def add_time_context_features(result: pd.DataFrame) -> pd.DataFrame:
    if result.empty:
        return result

    window_mid = result['window_start'] + (WINDOW_SIZE / 2)
    hour_of_day = window_mid.dt.hour + (window_mid.dt.minute / 60.0)
    day_of_week = window_mid.dt.dayofweek.astype('int64')
    day_of_year = window_mid.dt.dayofyear.astype('int64')
    month = window_mid.dt.month.astype('int64')

    result['hour_of_day'] = hour_of_day
    result['day_of_week'] = day_of_week
    result['day_of_year'] = day_of_year
    result['month'] = month
    result['is_weekend'] = day_of_week.isin([5, 6])
    result['is_heating_season'] = month.isin(sorted(HEATING_SEASON_MONTHS))
    result['season_bucket'] = month.map(season_bucket_from_month)

    result['hour_sin'] = np.sin(2 * np.pi * hour_of_day / 24.0)
    result['hour_cos'] = np.cos(2 * np.pi * hour_of_day / 24.0)
    result['dow_sin'] = np.sin(2 * np.pi * day_of_week / 7.0)
    result['dow_cos'] = np.cos(2 * np.pi * day_of_week / 7.0)
    result['doy_sin'] = np.sin(2 * np.pi * day_of_year / 366.0)
    result['doy_cos'] = np.cos(2 * np.pi * day_of_year / 366.0)
    return result


def prepare_operational_frame(path: Path) -> tuple[pd.DataFrame, list[str], list[str], int]:
    df = read_predist_csv(path)
    if 'timestamp' not in df.columns:
        raise ValueError(f'timestamp ??? ????: {path}')

    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    invalid_timestamp_rows = int(df['timestamp'].isna().sum())
    df = df.dropna(subset=['timestamp']).sort_values('timestamp')
    df = df.drop_duplicates(subset=['timestamp'], keep='last').reset_index(drop=True)

    numeric_columns = [column for column in CORE_SENSOR_COLUMNS if column in df.columns]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

    context_columns = [
        column for column in df.columns
        if column != 'timestamp' and CONTROL_CONTEXT_PATTERN.search(column)
    ]
    for column in context_columns:
        df[column] = df[column].astype('string').fillna('missing')

    return df, numeric_columns, context_columns, invalid_timestamp_rows


def estimate_expected_rows(df: pd.DataFrame) -> tuple[int, float]:
    diffs = df['timestamp'].diff().dropna()
    if diffs.empty:
        return 1, np.nan

    median_interval = diffs.median()
    if pd.isna(median_interval) or median_interval <= pd.Timedelta(0):
        return 1, np.nan

    expected_rows = max(1, int(math.ceil(WINDOW_SIZE / median_interval)))
    median_minutes = median_interval.total_seconds() / 60
    return expected_rows, median_minutes


def summarize_top_sensors(values: dict[str, float], limit: int = 5) -> str:
    valid_items = [(key, value) for key, value in values.items() if pd.notna(value) and value > 0]
    valid_items = sorted(valid_items, key=lambda item: item[1], reverse=True)
    return '|'.join(key for key, _ in valid_items[:limit])


def add_sensor_error_count(df: pd.DataFrame, filled: pd.DataFrame, numeric_columns: list[str]) -> pd.Series:
    error_count = pd.Series(0, index=df.index, dtype='int64')

    for column in TEMPERATURE_COLUMNS:
        if column in numeric_columns:
            error_count += ((filled[column] < -50) | (filled[column] > 150)).astype('int64')
    for column in FLOW_COLUMNS + POWER_COLUMNS:
        if column in numeric_columns:
            error_count += (filled[column] < 0).astype('int64')
    for column in CUMULATIVE_COLUMNS:
        if column in numeric_columns:
            error_count += (filled[column].groupby(df['window_start']).diff() < 0).astype('int64')

    return error_count.groupby(df['window_start']).sum()


def add_extreme_change_count(df: pd.DataFrame, filled: pd.DataFrame, numeric_columns: list[str]) -> pd.Series:
    total_extreme = pd.Series(0, index=df.index, dtype='int64')

    for column in numeric_columns:
        diff = filled[column].groupby(df['window_start']).diff().abs()
        baseline = diff.groupby(df['window_start']).transform('median')
        extreme = (baseline > 0) & (diff > baseline * 10)
        total_extreme += extreme.fillna(False).astype('int64')

    return total_extreme.groupby(df['window_start']).sum()


def get_relevant_intervals(manufacturer: str, substation_id: int) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    intervals = []

    fault_rows = fault_alignment[
        (fault_alignment['manufacturer'] == manufacturer)
        & (fault_alignment['substation_id'] == substation_id)
    ]
    max_prefault_lead = pd.Timedelta(hours=MAX_PREFAULT_LEAD_HOURS)
    for row in fault_rows.itertuples(index=False):
        if pd.isna(row.window_start) or pd.isna(row.window_end):
            continue

        start = row.window_start
        end = row.window_end
        if pd.notna(row.report_date):
            start = max(start, row.report_date - max_prefault_lead)
            end = min(end, row.report_date)

        if pd.notna(start) and pd.notna(end) and start < end:
            intervals.append((start, end))

    normal_rows = normal_alignment[
        (normal_alignment['manufacturer'] == manufacturer)
        & (normal_alignment['substation_id'] == substation_id)
    ]
    for row in normal_rows.itertuples(index=False):
        if pd.notna(row.window_start) and pd.notna(row.window_end):
            intervals.append((row.window_start, row.window_end))

    disturbance_rows = disturbance_alignment[
        (disturbance_alignment['manufacturer'] == manufacturer)
        & (disturbance_alignment['substation_id'] == substation_id)
    ]
    context = pd.Timedelta(hours=DISTURBANCE_CONTEXT_HOURS)
    for row in disturbance_rows.itertuples(index=False):
        if pd.notna(row.event_start):
            intervals.append((row.event_start - context, row.event_start + context))

    return intervals


def filter_relevant_rows(df: pd.DataFrame, intervals: list[tuple[pd.Timestamp, pd.Timestamp]]) -> pd.DataFrame:
    if not KEEP_ONLY_RELEVANT_WINDOWS:
        return df
    if not intervals:
        return df.iloc[0:0].copy()

    mask = pd.Series(False, index=df.index)
    for start, end in intervals:
        mask |= (df['timestamp'] >= start) & (df['timestamp'] < end)
    return df.loc[mask].copy()


def summarize_context_mode(series: pd.Series) -> str:
    modes = series.mode(dropna=False)
    if modes.empty:
        return 'missing'
    return str(modes.iloc[0])


def build_window_features(path: Path) -> pd.DataFrame:
    manufacturer = path.parents[1].name
    substation_id = extract_substation_id(path)
    df, numeric_columns, context_columns, invalid_timestamp_rows = prepare_operational_frame(path)

    if df.empty or not numeric_columns:
        return pd.DataFrame()

    expected_rows, median_interval_minutes = estimate_expected_rows(df)
    relevant_intervals = get_relevant_intervals(manufacturer, substation_id)
    df = filter_relevant_rows(df, relevant_intervals)
    if df.empty:
        return pd.DataFrame()

    df['window_start'] = df['timestamp'].dt.floor(WINDOW_FREQ)
    grouped = df.groupby('window_start', sort=True)

    result = pd.DataFrame(index=grouped.size().index)
    result['manufacturer'] = manufacturer
    result['substation_id'] = substation_id
    result['source_file'] = path.name
    result['window_start'] = result.index
    result['window_end'] = result['window_start'] + WINDOW_SIZE
    result['row_count'] = grouped.size().astype('int64')
    result['expected_row_count'] = int(expected_rows)
    result['median_interval_minutes'] = median_interval_minutes
    result['invalid_timestamp_rows_in_file'] = invalid_timestamp_rows

    timestamp_diffs = df.groupby('window_start')['timestamp'].diff()
    timestamp_gap_minutes = timestamp_diffs.dt.total_seconds() / 60
    if pd.notna(median_interval_minutes):
        result['timestamp_gap_count'] = (
            (timestamp_gap_minutes > median_interval_minutes * 1.5)
            .groupby(df['window_start'])
            .sum()
            .reindex(result.index, fill_value=0)
            .astype('int64')
        )
    else:
        result['timestamp_gap_count'] = 0
    result['max_timestamp_gap_minutes'] = (
        timestamp_gap_minutes
        .groupby(df['window_start'])
        .max()
        .reindex(result.index)
        .fillna(0)
    )

    raw_numeric = df[numeric_columns]
    filled = raw_numeric.groupby(df['window_start']).ffill()
    filled = filled.groupby(df['window_start']).bfill()
    window_medians = raw_numeric.groupby(df['window_start']).transform('median')
    filled = filled.fillna(window_medians)
    missing_mask = raw_numeric.isna()

    result['missing_count'] = (
        missing_mask.sum(axis=1)
        .groupby(df['window_start'])
        .sum()
        .reindex(result.index, fill_value=0)
        .astype('int64')
    )
    total_values = result['row_count'] * len(numeric_columns)
    result['missing_rate'] = result['missing_count'] / total_values.replace(0, np.nan)
    result['sensor_error_candidate_count'] = add_sensor_error_count(df, filled, numeric_columns).reindex(result.index, fill_value=0).astype('int64')
    result['extreme_change_count'] = add_extreme_change_count(df, filled, numeric_columns).reindex(result.index, fill_value=0).astype('int64')

    missing_sum = missing_mask.groupby(df['window_start']).sum().reindex(result.index, fill_value=0)
    missing_rate = missing_mask.groupby(df['window_start']).mean().reindex(result.index, fill_value=0)

    for column in numeric_columns:
        stats = filled[column].groupby(df['window_start']).agg(['mean', 'min', 'max', 'std', 'first', 'last']).reindex(result.index)
        result[f'{column}__mean'] = stats['mean']
        result[f'{column}__min'] = stats['min']
        result[f'{column}__max'] = stats['max']
        result[f'{column}__std'] = stats['std']
        result[f'{column}__first'] = stats['first']
        result[f'{column}__last'] = stats['last']
        result[f'{column}__delta'] = stats['last'] - stats['first']
        result[f'{column}__missing_count'] = missing_sum[column].astype('int64')
        result[f'{column}__missing_rate'] = missing_rate[column]

    for feature_name, (left_column, right_column) in DERIVED_PAIRS.items():
        if left_column in filled.columns and right_column in filled.columns:
            diff = filled[left_column] - filled[right_column]
            diff_grouped = diff.groupby(df['window_start'])
            result[f'{feature_name}__mean'] = diff_grouped.mean().reindex(result.index)
            result[f'{feature_name}__max_abs'] = diff.abs().groupby(df['window_start']).max().reindex(result.index)
            result[f'{feature_name}__last'] = diff_grouped.last().reindex(result.index)

    for column in context_columns:
        grouped_context = df.groupby('window_start')[column]
        result[f'{column}__dominant'] = grouped_context.agg(summarize_context_mode).reindex(result.index).fillna('missing')
        result[f'{column}__nunique'] = grouped_context.nunique(dropna=False).reindex(result.index, fill_value=0).astype('int64')
        result[f'{column}__change_count'] = (
            df[column].ne(df.groupby('window_start')[column].shift())
            .groupby(df['window_start'])
            .sum()
            .sub(1)
            .clip(lower=0)
            .reindex(result.index, fill_value=0)
            .astype('int64')
        )

    result['main_missing_sensors'] = missing_sum.apply(lambda row: summarize_top_sensors(row.to_dict()), axis=1)
    delta_values = pd.DataFrame({
        column: result[f'{column}__delta'].abs() for column in numeric_columns if f'{column}__delta' in result.columns
    })
    result['main_changed_sensors'] = delta_values.apply(lambda row: summarize_top_sensors(row.to_dict()), axis=1)
    result['data_quality_issue'] = (
        (result['row_count'] < expected_rows * MIN_ROW_RATIO)
        | (result['missing_rate'] > MAX_MISSING_RATE)
        | (result['sensor_error_candidate_count'] > 0)
    )

    result = add_time_context_features(result)
    return result.reset_index(drop=True)


## 4. 라벨 연결 함수

`pre_fault` 라벨은 fault 구간과 윈도우가 겹치면서, 윈도우 종료 시점이 `report_date` 이전인 경우에만 붙인다.

또한 초기 학습 후보에서는 고장신고 전 7일 이내 구간만 `pre_fault`로 사용한다.
일부 fault는 `Possible anomaly start/end`가 고장신고 이후까지 포함될 수 있으므로, 03번에서도 fault 구간을 `report_date` 이전으로 한 번 더 자른다.

`unlabeled`는 정상 라벨이 아니다. 학습에 바로 쓰지 않도록 `window_source_type`, `use_for_supervised_training` 컬럼을 함께 만든다.

In [5]:

def overlaps(window_start: pd.Timestamp, window_end: pd.Timestamp, event_start: pd.Timestamp, event_end: pd.Timestamp) -> bool:
    if pd.isna(event_start) or pd.isna(event_end):
        return False
    return window_start < event_end and window_end > event_start


def clamp_fault_window(fault) -> tuple[pd.Timestamp, pd.Timestamp]:
    start = fault.window_start
    end = fault.window_end
    if pd.notna(fault.report_date):
        max_prefault_lead = pd.Timedelta(hours=MAX_PREFAULT_LEAD_HOURS)
        start = max(start, fault.report_date - max_prefault_lead)
        end = min(end, fault.report_date)
    return start, end


def classify_window_source(label: str, normal_related: bool, maintenance_related: bool, blocked_fault_count: int) -> str:
    if label == 'pre_fault':
        return 'pre_fault_context'
    if label == 'normal':
        return 'normal_context'
    if blocked_fault_count > 0:
        return 'post_fault_blocked'
    if maintenance_related:
        return 'disturbance_context'
    if normal_related:
        return 'normal_overlap_unlabeled'
    return 'related_unlabeled'


def latest_past_event_days(event_times: pd.Series, window_start: pd.Timestamp) -> float:
    if pd.isna(window_start):
        return np.nan
    earlier_events = event_times.loc[event_times < window_start]
    if earlier_events.empty:
        return np.nan
    latest_event = earlier_events.max()
    return float((window_start - latest_event).total_seconds() / 86400.0)


def attach_labels(windows: pd.DataFrame) -> pd.DataFrame:
    if windows.empty:
        return windows

    result = windows.copy()
    manufacturer = result['manufacturer'].iloc[0]
    substation_id = result['substation_id'].iloc[0]

    fault_candidates = fault_alignment[
        (fault_alignment['manufacturer'] == manufacturer)
        & (fault_alignment['substation_id'] == substation_id)
    ]
    normal_candidates = normal_alignment[
        (normal_alignment['manufacturer'] == manufacturer)
        & (normal_alignment['substation_id'] == substation_id)
    ]
    disturbance_candidates = disturbance_alignment[
        (disturbance_alignment['manufacturer'] == manufacturer)
        & (disturbance_alignment['substation_id'] == substation_id)
    ]
    prior_fault_times = fault_candidates['report_date'].dropna() if 'report_date' in fault_candidates.columns else pd.Series(dtype='datetime64[ns]')
    prior_task_times = disturbance_candidates['event_start'].dropna() if 'event_start' in disturbance_candidates.columns else pd.Series(dtype='datetime64[ns]')

    label_rows = []

    for row in result.itertuples(index=False):
        window_start = row.window_start
        window_end = row.window_end

        matched_faults = []
        leakage_blocked_count = 0
        for fault in fault_candidates.itertuples(index=False):
            original_overlaps = overlaps(window_start, window_end, fault.window_start, fault.window_end)
            fault_start, fault_end = clamp_fault_window(fault)
            clamped_overlaps = overlaps(window_start, window_end, fault_start, fault_end)

            if clamped_overlaps and pd.notna(fault.report_date) and window_end <= fault.report_date:
                lead_time = (fault.report_date - window_end).total_seconds() / 3600
                if 0 <= lead_time <= MAX_PREFAULT_LEAD_HOURS:
                    matched_faults.append((lead_time, fault))
            elif original_overlaps:
                leakage_blocked_count += 1

        matched_normals = [
            normal for normal in normal_candidates.itertuples(index=False)
            if overlaps(window_start, window_end, normal.window_start, normal.window_end)
        ]
        matched_disturbances = [
            disturbance for disturbance in disturbance_candidates.itertuples(index=False)
            if pd.notna(disturbance.event_start) and window_start <= disturbance.event_start < window_end
        ]

        label = 'unlabeled'
        fault_label = np.nan
        fault_event_id = np.nan
        estimated_lead_time_hours = np.nan

        if matched_normals:
            label = 'normal'
        if matched_faults:
            matched_faults = sorted(matched_faults, key=lambda item: item[0])
            estimated_lead_time_hours, selected_fault = matched_faults[0]
            label = 'pre_fault'
            fault_label = selected_fault.fault_label
            fault_event_id = selected_fault.event_id

        normal_related = bool(matched_normals)
        maintenance_related = bool(matched_disturbances)
        window_source_type = classify_window_source(
            label,
            normal_related,
            maintenance_related,
            leakage_blocked_count,
        )

        days_since_last_fault_event = latest_past_event_days(prior_fault_times, window_start)
        days_since_last_task_event = latest_past_event_days(prior_task_times, window_start)
        valid_days = [value for value in [days_since_last_fault_event, days_since_last_task_event] if pd.notna(value)]
        days_since_last_any_event = min(valid_days) if valid_days else np.nan
        post_fault_stabilization = bool(pd.notna(days_since_last_fault_event) and days_since_last_fault_event <= STABILIZATION_DAYS)
        post_task_stabilization = bool(pd.notna(days_since_last_task_event) and days_since_last_task_event <= STABILIZATION_DAYS)

        label_rows.append({
            'label': label,
            'fault_label': fault_label,
            'fault_event_id': fault_event_id,
            'estimated_lead_time_hours': estimated_lead_time_hours,
            'normal_event_related': normal_related,
            'maintenance_related': maintenance_related,
            'disturbance_count': len(matched_disturbances),
            'leakage_blocked_fault_count': leakage_blocked_count,
            'window_source_type': window_source_type,
            'use_for_supervised_training': label in {'normal', 'pre_fault'},
            'days_since_last_fault_event': days_since_last_fault_event,
            'days_since_last_task_event': days_since_last_task_event,
            'days_since_last_any_event': days_since_last_any_event,
            'post_fault_stabilization': post_fault_stabilization,
            'post_task_stabilization': post_task_stabilization,
            'recent_regime_change_flag': post_fault_stabilization or post_task_stabilization,
        })

    return pd.concat([result.reset_index(drop=True), pd.DataFrame(label_rows)], axis=1)


def attach_configuration(windows: pd.DataFrame) -> pd.DataFrame:
    if windows.empty:
        return windows
    result = windows.merge(
        configuration_table,
        on=['manufacturer', 'substation_id'],
        how='left',
    )
    result['normal_reference_group'] = (
        result['manufacturer'].fillna('missing')
        + '|'
        + result['configuration_type'].fillna('missing')
        + '|'
        + result['season_bucket'].fillna('missing')
    )
    return result


## 5. 전체 운영 파일 처리

각 운영 CSV를 읽어서 윈도우 feature를 만들고, 라벨과 설비 구성을 붙인다.

초기 기본값은 전체 운영 기간이 아니라 `fault`, `normal`, `disturbance`와 연결되는 분석 필요 구간만 처리한다.
이렇게 하는 이유는 라벨 없는 장기 구간까지 모두 처리하면 03번이 너무 무거워지고, 초기 supervised baseline에 바로 쓰기 어렵기 때문이다.

나중에 운영 전체 기간 추론용 데이터가 필요하면 `KEEP_ONLY_RELEVANT_WINDOWS = False`로 바꿔 전체 윈도우를 생성할 수 있다.

In [6]:
operational_files = list_operational_files(RAW_ROOT)
print(f'?? ?? ?: {len(operational_files)}')
operational_files[:3]

?? ?? ?: 93


[WindowsPath('C:/Project3/HeatGrid_Agent/data/raw_data/predist_v2/manufacturer 1/operational_data/substation_1.csv'),
 WindowsPath('C:/Project3/HeatGrid_Agent/data/raw_data/predist_v2/manufacturer 1/operational_data/substation_10.csv'),
 WindowsPath('C:/Project3/HeatGrid_Agent/data/raw_data/predist_v2/manufacturer 1/operational_data/substation_11.csv')]

In [7]:
window_frames = []
failed_files = []

for index, path in enumerate(operational_files, start=1):
    try:
        windows = build_window_features(path)
        windows = attach_labels(windows)
        windows = attach_configuration(windows)
        if not windows.empty:
            window_frames.append(windows)
        if index % 10 == 0 or index == len(operational_files):
            print(f'처리 진행: {index}/{len(operational_files)}')
    except Exception as error:
        failed_files.append({'file': str(path), 'error': str(error)})

if window_frames:
    ml_window_dataset = pd.concat(window_frames, ignore_index=True)
else:
    ml_window_dataset = pd.DataFrame()

print(f'생성된 윈도우 행 수: {len(ml_window_dataset):,}')
print(f'처리 실패 파일 수: {len(failed_files)}')

if failed_files:
    pd.DataFrame(failed_files)

C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:259: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__delta'] = stats['last'] - stats['first']
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:260: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__missing_count'] = missing_sum[column].astype('int64')
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:261: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse

C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:259: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__delta'] = stats['last'] - stats['first']
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:260: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__missing_count'] = missing_sum[column].astype('int64')
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:261: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse

C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:259: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__delta'] = stats['last'] - stats['first']
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:260: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__missing_count'] = missing_sum[column].astype('int64')
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:261: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse

C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:259: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__delta'] = stats['last'] - stats['first']
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:260: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__missing_count'] = missing_sum[column].astype('int64')
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:261: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse

C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:259: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__delta'] = stats['last'] - stats['first']
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:260: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__missing_count'] = missing_sum[column].astype('int64')
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:261: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse

...

C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:259: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__delta'] = stats['last'] - stats['first']
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:260: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__missing_count'] = missing_sum[column].astype('int64')
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:261: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse

처리 진행: 90/93


C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:259: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__delta'] = stats['last'] - stats['first']
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:260: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__missing_count'] = missing_sum[column].astype('int64')
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:261: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse

처리 진행: 93/93
생성된 윈도우 행 수: 3,270
처리 실패 파일 수: 0


C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:259: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__delta'] = stats['last'] - stats['first']
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:260: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f'{column}__missing_count'] = missing_sum[column].astype('int64')
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\1438684591.py:261: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse

## 6. holdout 분리 컬럼 생성

03번에서 `train`, `validation`, `holdout` 컬럼을 미리 만들어 둔다.

- `split_time_based`: 제조사별 시간 순서를 기준으로 분리
- `split_substation_based`: 기계실을 통째로 분리

초기 모델링에서는 `split_time_based`를 기본으로 사용하고, 이후 기계실 일반화 성능을 볼 때 `split_substation_based`를 비교한다.

In [8]:

def assign_time_based_split(df: pd.DataFrame) -> pd.Series:
    split = pd.Series(index=df.index, dtype='object')
    for manufacturer, group in df.groupby('manufacturer'):
        sorted_index = group.sort_values('window_start').index.to_list()
        n_rows = len(sorted_index)
        train_end = int(n_rows * 0.70)
        validation_end = int(n_rows * 0.85)
        split.loc[sorted_index[:train_end]] = 'train'
        split.loc[sorted_index[train_end:validation_end]] = 'validation'
        split.loc[sorted_index[validation_end:]] = 'holdout'
    return split


def hash_to_ratio(value: str) -> float:
    digest = hashlib.md5(value.encode('utf-8')).hexdigest()
    return int(digest[:8], 16) / 0xFFFFFFFF


def assign_substation_based_split(df: pd.DataFrame) -> pd.Series:
    split_map = {}
    keys = df[['manufacturer', 'substation_id']].drop_duplicates()
    for key in keys.itertuples(index=False):
        ratio = hash_to_ratio(f'{key.manufacturer}_{key.substation_id}')
        if ratio < 0.70:
            split_map[(key.manufacturer, key.substation_id)] = 'train'
        elif ratio < 0.85:
            split_map[(key.manufacturer, key.substation_id)] = 'validation'
        else:
            split_map[(key.manufacturer, key.substation_id)] = 'holdout'
    return df.apply(lambda row: split_map[(row['manufacturer'], row['substation_id'])], axis=1)


def assign_regime_based_split(df: pd.DataFrame) -> pd.Series:
    split = pd.Series(index=df.index, dtype='object')
    group_columns = [
        column for column in ['manufacturer', 'configuration_type', 'season_bucket']
        if column in df.columns
    ]
    if not group_columns:
        return assign_time_based_split(df)

    for _, group in df.groupby(group_columns, dropna=False):
        sorted_index = group.sort_values('window_start').index.to_list()
        n_rows = len(sorted_index)
        if n_rows < 8:
            split.loc[sorted_index] = assign_time_based_split(group).reindex(sorted_index).to_list()
            continue
        train_end = int(n_rows * 0.70)
        validation_end = int(n_rows * 0.85)
        split.loc[sorted_index[:train_end]] = 'train'
        split.loc[sorted_index[train_end:validation_end]] = 'validation'
        split.loc[sorted_index[validation_end:]] = 'holdout'
    return split


if not ml_window_dataset.empty:
    ml_window_dataset['split_time_based'] = assign_time_based_split(ml_window_dataset)
    ml_window_dataset['split_substation_based'] = assign_substation_based_split(ml_window_dataset)
    ml_window_dataset['split_regime_based'] = assign_regime_based_split(ml_window_dataset)

split_summary = pd.DataFrame()
if not ml_window_dataset.empty:
    split_summary = (
        ml_window_dataset
        .groupby(['split_regime_based', 'label'])
        .size()
        .reset_index(name='??? ?')
        .rename(columns={'split_regime_based': 'regime ?? ??', 'label': '??'})
    )
split_summary


,regime ?? ??,??,??? ?
0,holdout,normal,244
1,holdout,pre_fault,126
2,holdout,unlabeled,132
3,train,normal,1272
4,train,pre_fault,579
5,train,unlabeled,427
6,validation,normal,302
7,validation,pre_fault,110
8,validation,unlabeled,78


## 6-A. normal reference filter

06 audit ???? `manufacturer 2 / SH` normal? train reference? ?? ??? ????.
??? 03??? split ?? ? train normal ?? IQR ???? ?? ???? normal window? ?? ????,
?? window? supervised training ?? ???? ????.


In [9]:

NORMAL_REFERENCE_TARGET = {
    'manufacturer': 'manufacturer 2',
    'configuration_type': 'SH',
}
NORMAL_REFERENCE_FEATURES = [
    'p_net_return_temperature__std',
    'p_net_return_temperature__mean',
    's_hc1_supply_temperature_setpoint__std',
]
NORMAL_REFERENCE_MIN_OUTLIER_FEATURES = 2


def apply_normal_reference_filter(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    result = df.copy()
    result['normal_reference_outlier'] = False
    result['normal_reference_outlier_count'] = 0
    result['normal_reference_filter_reason'] = ''

    target_mask = (
        result['label'].eq('normal')
        & result['manufacturer'].eq(NORMAL_REFERENCE_TARGET['manufacturer'])
        & result['configuration_type'].eq(NORMAL_REFERENCE_TARGET['configuration_type'])
    )
    if 'post_fault_stabilization' in result.columns:
        target_mask &= ~result['post_fault_stabilization'].fillna(False)
    if 'post_task_stabilization' in result.columns:
        target_mask &= ~result['post_task_stabilization'].fillna(False)

    train_reference_mask = target_mask & result['split_regime_based'].eq('train')
    available_features = [column for column in NORMAL_REFERENCE_FEATURES if column in result.columns]

    threshold_rows = []
    if train_reference_mask.sum() == 0 or not available_features:
        return result, pd.DataFrame(threshold_rows)

    outlier_counts = pd.Series(0, index=result.index, dtype='int64')
    outlier_reasons = {index: [] for index in result.index}

    for column in available_features:
        train_series = result.loc[train_reference_mask, column]
        q1 = train_series.quantile(0.25)
        q3 = train_series.quantile(0.75)
        iqr = q3 - q1
        low = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        threshold_rows.append({
            'manufacturer': NORMAL_REFERENCE_TARGET['manufacturer'],
            'configuration_type': NORMAL_REFERENCE_TARGET['configuration_type'],
            'feature': column,
            'train_q1': q1,
            'train_q3': q3,
            'train_iqr': iqr,
            'low_threshold': low,
            'high_threshold': high,
        })
        feature_mask = target_mask & ((result[column] < low) | (result[column] > high))
        outlier_counts.loc[feature_mask] += 1
        for index in result.index[feature_mask]:
            outlier_reasons[index].append(column)

    result['normal_reference_outlier_count'] = outlier_counts
    result['normal_reference_filter_reason'] = result.index.to_series().map(lambda index: '|'.join(outlier_reasons[index]))
    result['normal_reference_outlier'] = target_mask & result['normal_reference_outlier_count'].ge(NORMAL_REFERENCE_MIN_OUTLIER_FEATURES)
    if 'post_fault_stabilization' in result.columns:
        result.loc[result['post_fault_stabilization'].fillna(False), 'use_for_supervised_training'] = False
    if 'post_task_stabilization' in result.columns:
        result.loc[result['post_task_stabilization'].fillna(False), 'use_for_supervised_training'] = False
    result.loc[result['normal_reference_outlier'], 'use_for_supervised_training'] = False
    return result, pd.DataFrame(threshold_rows)


normal_reference_thresholds = pd.DataFrame()
if not ml_window_dataset.empty:
    ml_window_dataset, normal_reference_thresholds = apply_normal_reference_filter(ml_window_dataset)

display(normal_reference_thresholds)
display(
    ml_window_dataset.loc[
        ml_window_dataset['label'].eq('normal')
        & ml_window_dataset['manufacturer'].eq(NORMAL_REFERENCE_TARGET['manufacturer'])
        & ml_window_dataset['configuration_type'].eq(NORMAL_REFERENCE_TARGET['configuration_type'])
    ]
    .groupby('split_regime_based')[['normal_reference_outlier', 'use_for_supervised_training']]
    .mean()
)


,manufacturer,configuration_type,feature,train_q1,train_q3,train_iqr,low_threshold,high_threshold
0,manufacturer 2,SH,p_net_return_temperature__std,1.416457,2.739917,1.323460,-0.568733,4.725107
1,manufacturer 2,SH,p_net_return_temperature__mean,49.583333,61.750000,12.166667,31.333333,80.000000
2,manufacturer 2,SH,s_hc1_supply_temperature_setpoint__std,0.000000,1.378437,1.378437,-2.067655,3.446091


,normal_reference_outlier,use_for_supervised_training
split_regime_based,,
holdout,0.116667,0.883333
train,0.000000,0.866029
validation,0.153846,0.846154


## 7. 결과 요약 확인

여기서는 전체 데이터를 길게 출력하지 않는다.

03번에서 확인할 핵심은 다음이다.

- 몇 개의 윈도우가 만들어졌는지
- 라벨 분포가 어떻게 되는지
- 데이터 품질 문제가 많은지
- 고장 전 위험 후보의 lead time 범위가 어떤지

In [10]:
if ml_window_dataset.empty:
    raise RuntimeError('생성된 윈도우 데이터셋이 없습니다.')

result_summary = pd.DataFrame([
    {'항목': '전체 윈도우 수', '값': len(ml_window_dataset)},
    {'항목': '기계실 수', '값': ml_window_dataset[['manufacturer', 'substation_id']].drop_duplicates().shape[0]},
    {'항목': 'feature 포함 컬럼 수', '값': ml_window_dataset.shape[1]},
    {'항목': '데이터 품질 이슈 윈도우 수', '값': int(ml_window_dataset['data_quality_issue'].sum())},
    {'항목': '센서 오류 후보 포함 윈도우 수', '값': int((ml_window_dataset['sensor_error_candidate_count'] > 0).sum())},
])
result_summary

,항목,값
0,전체 윈도우 수,3270
1,기계실 수,69
2,feature 포함 컬럼 수,252
3,데이터 품질 이슈 윈도우 수,337
4,센서 오류 후보 포함 윈도우 수,34


In [11]:
label_summary = (
    ml_window_dataset
    .groupby('label')
    .size()
    .reset_index(name='윈도우 수')
    .rename(columns={'label': '라벨'})
    .sort_values('윈도우 수', ascending=False)
)
label_summary

,라벨,윈도우 수
0,normal,1818
1,pre_fault,815
2,unlabeled,637


In [12]:
lead_time_summary = ml_window_dataset.loc[
    ml_window_dataset['label'] == 'pre_fault',
    'estimated_lead_time_hours'
].describe()

lead_time_summary_table = lead_time_summary.reset_index()
lead_time_summary_table.columns = ['통계', '고장신고까지 남은 시간']
lead_time_summary_table

,통계,고장신고까지 남은 시간
0,count,815.000000
1,mean,41.582293
2,std,32.062968
3,min,0.066667
4,25%,16.633333
5,50%,37.250000
6,75%,57.758333
7,max,167.533333


## 7-A. normal 기준 보강용 profile

06 audit에서 holdout normal 분포 차이가 확인됐으므로, 03번에서도 normal 행을 제조사와 설비 구성 기준으로 요약한다.
이 표는 이후 normal 기준 재검토 때 어떤 group이 train 기준 분포에서 멀어지는지 확인하는 기준점으로 쓴다.


In [13]:
normal_profile_columns = [
    'p_net_return_temperature__mean',
    'p_net_return_temperature__max',
    'p_net_return_temperature__std',
    'p_net_supply_temperature__std',
    'network_temperature_gap__mean',
    'network_temperature_gap__last',
    's_hc1_supply_temperature_setpoint__std',
]
normal_profile_columns = [column for column in normal_profile_columns if column in ml_window_dataset.columns]

normal_windows = ml_window_dataset.loc[ml_window_dataset['label'] == 'normal'].copy()
normal_profile_by_group = pd.DataFrame()
normal_reference_drift_by_group = pd.DataFrame()

if not normal_windows.empty:
    group_columns = ['split_time_based', 'manufacturer', 'configuration_type']
    aggregate_map = {
        'substation_id': pd.Series.nunique,
        'maintenance_related': 'mean',
        'data_quality_issue': 'mean',
    }
    for column in normal_profile_columns:
        aggregate_map[column] = 'mean'

    normal_profile_by_group = (
        normal_windows
        .groupby(group_columns, dropna=False)
        .agg(aggregate_map)
        .rename(columns={
            'substation_id': 'substation_count',
            'maintenance_related': 'maintenance_related_rate',
            'data_quality_issue': 'data_quality_issue_rate',
        })
        .reset_index()
    )

    drift_rows = []
    reference_keys = ['manufacturer', 'configuration_type']
    for key, group in normal_windows.groupby(reference_keys, dropna=False):
        train_group = group.loc[group['split_time_based'] == 'train']
        holdout_group = group.loc[group['split_time_based'] == 'holdout']
        if train_group.empty or holdout_group.empty:
            continue
        manufacturer, configuration_type = key
        for column in normal_profile_columns:
            train_mean = train_group[column].mean()
            holdout_mean = holdout_group[column].mean()
            train_std = train_group[column].std()
            if pd.isna(train_std) or train_std == 0:
                train_std = 1.0
            drift_rows.append({
                'manufacturer': manufacturer,
                'configuration_type': configuration_type,
                'feature': column,
                'train_rows': len(train_group),
                'holdout_rows': len(holdout_group),
                'train_mean': train_mean,
                'holdout_mean': holdout_mean,
                'holdout_z_vs_train': abs((holdout_mean - train_mean) / train_std),
            })

    normal_reference_drift_by_group = pd.DataFrame(drift_rows)
    if not normal_reference_drift_by_group.empty:
        normal_reference_drift_by_group = normal_reference_drift_by_group.sort_values(
            ['holdout_z_vs_train', 'manufacturer', 'configuration_type'],
            ascending=[False, True, True],
        )

display(normal_profile_by_group.head(20))
display(normal_reference_drift_by_group.head(20))


,split_time_based,manufacturer,configuration_type,substation_count,maintenance_related_rate,data_quality_issue_rate,p_net_return_temperature__mean,p_net_return_temperature__max,p_net_return_temperature__std,p_net_supply_temperature__std,network_temperature_gap__mean,network_temperature_gap__last,s_hc1_supply_temperature_setpoint__std
0,holdout,manufacturer 1,SH + DHW,3,0.0,0.071429,38.648988,46.250000,3.821065,3.057981,40.886501,40.885516,5.307139
1,holdout,manufacturer 1,SH + DHW with sub-circuits,1,0.0,0.000000,45.652609,50.201429,2.812256,0.561134,48.661300,49.652143,3.799958
2,holdout,manufacturer 2,SH,2,0.0,0.000000,39.448909,47.214286,4.476250,1.790380,45.352183,45.625000,3.185106
3,holdout,manufacturer 2,SH + DHW,4,0.0,0.000000,52.542849,57.042553,3.172460,0.996682,34.550532,34.436170,3.530082
4,holdout,manufacturer 2,SH with buffer tank,1,0.0,0.000000,53.722222,58.800000,3.557775,1.162146,50.388889,49.800000,1.553590
5,train,manufacturer 1,SH + DHW,12,0.0,0.031835,49.013523,51.853470,1.916237,1.012418,38.586629,38.532961,2.180681
6,train,manufacturer 1,SH + DHW with sub-circuits,4,0.0,0.000000,49.809658,54.065980,3.350100,0.445523,40.003375,39.165850,3.877015
7,train,manufacturer 2,SH,4,0.0,0.000000,56.599348,60.045918,1.953771,0.990516,36.425028,36.280612,0.771222
8,train,manufacturer 2,SH + DHW,6,0.0,0.000000,54.432359,58.272727,3.863951,1.006646,24.251984,24.262987,3.000101
9,train,manufacturer 2,SH with buffer tank,2,0.0,0.000000,52.385147,56.489796,2.810959,2.112902,44.068157,43.877551,0.655146


,manufacturer,configuration_type,feature,train_rows,holdout_rows,train_mean,holdout_mean,holdout_z_vs_train
16,manufacturer 2,SH,p_net_return_temperature__std,196,56,1.953771,4.476250,3.150134
14,manufacturer 2,SH,p_net_return_temperature__mean,196,56,56.599348,39.448909,2.649238
20,manufacturer 2,SH,s_hc1_supply_temperature_setpoint__std,196,56,0.771222,3.185106,2.622893
15,manufacturer 2,SH,p_net_return_temperature__max,196,56,60.045918,47.214286,2.243908
32,manufacturer 2,SH with buffer tank,network_temperature_gap__mean,49,5,44.068157,50.388889,2.099093
12,manufacturer 1,SH + DHW with sub-circuits,network_temperature_gap__last,165,28,39.165850,49.652143,1.504010
3,manufacturer 1,SH + DHW,p_net_supply_temperature__std,534,84,1.012418,3.057981,1.425462
29,manufacturer 2,SH with buffer tank,p_net_return_temperature__max,49,5,56.489796,58.800000,1.297178
11,manufacturer 1,SH + DHW with sub-circuits,network_temperature_gap__mean,165,28,40.003375,48.661300,1.289963
33,manufacturer 2,SH with buffer tank,network_temperature_gap__last,49,5,43.877551,49.800000,1.208251


In [14]:
display_columns = [
    'manufacturer',
    'substation_id',
    'window_start',
    'window_end',
    'label',
    'fault_label',
    'estimated_lead_time_hours',
    'maintenance_related',
    'data_quality_issue',
    'main_changed_sensors',
    'window_source_type',
    'use_for_supervised_training',
    'split_time_based',
]

sample_preview = ml_window_dataset[display_columns].head(10).rename(columns={
    'manufacturer': '제조사',
    'substation_id': '기계실 ID',
    'window_start': '윈도우 시작',
    'window_end': '윈도우 종료',
    'label': '라벨',
    'fault_label': '고장 라벨',
    'estimated_lead_time_hours': '고장신고까지 남은 시간',
    'maintenance_related': '정비/작업 관련',
    'data_quality_issue': '데이터 품질 이슈',
    'main_changed_sensors': '주요 변화 센서',
    'window_source_type': '윈도우 출처 유형',
    'use_for_supervised_training': '지도학습 사용 여부',
    'split_time_based': '시간 기준 분리',
})
sample_preview

,제조사,기계실 ID,윈도우 시작,윈도우 종료,라벨,고장 라벨,고장신고까지 남은 시간,정비/작업 관련,데이터 품질 이슈,주요 변화 센서,윈도우 출처 유형,지도학습 사용 여부,시간 기준 분리
0,manufacturer 1,1,2019-12-01 00:00:00,2019-12-01 06:00:00,normal,NaN,NaN,False,False,p_net_meter_energy|p_net_meter_flow|p_net_supp...,normal_context,True,validation
1,manufacturer 1,1,2019-12-01 06:00:00,2019-12-01 12:00:00,normal,NaN,NaN,False,False,p_net_meter_flow|p_net_meter_energy|p_net_mete...,normal_context,True,validation
2,manufacturer 1,1,2019-12-01 12:00:00,2019-12-01 18:00:00,normal,NaN,NaN,False,True,p_net_meter_energy|p_net_meter_flow|s_dhw_uppe...,normal_context,True,validation
3,manufacturer 1,1,2019-12-01 18:00:00,2019-12-02 00:00:00,normal,NaN,NaN,False,False,p_net_meter_flow|p_net_meter_energy|p_net_mete...,normal_context,True,validation
4,manufacturer 1,1,2019-12-02 00:00:00,2019-12-02 06:00:00,normal,NaN,NaN,False,True,p_net_meter_flow|p_net_meter_energy|p_net_mete...,normal_context,True,validation
5,manufacturer 1,1,2019-12-02 06:00:00,2019-12-02 12:00:00,normal,NaN,NaN,False,True,p_net_meter_flow|p_net_meter_energy|p_net_mete...,normal_context,True,validation
6,manufacturer 1,1,2019-12-02 12:00:00,2019-12-02 18:00:00,normal,NaN,NaN,False,False,p_net_meter_flow|p_net_meter_energy|p_net_mete...,normal_context,True,validation
7,manufacturer 1,1,2019-12-02 18:00:00,2019-12-03 00:00:00,normal,NaN,NaN,False,False,p_net_meter_flow|p_net_meter_energy|p_net_mete...,normal_context,True,validation
8,manufacturer 1,1,2019-12-03 00:00:00,2019-12-03 06:00:00,normal,NaN,NaN,False,False,p_net_meter_flow|p_net_meter_energy|s_dhw_lowe...,normal_context,True,validation
9,manufacturer 1,1,2019-12-03 06:00:00,2019-12-03 12:00:00,normal,NaN,NaN,False,False,p_net_meter_energy|p_net_meter_flow|s_dhw_uppe...,normal_context,True,validation


## 8. CSV 저장

저장된 CSV는 이후 단계에서 사용한다.

- 04번: feature 후보 정리 및 모델 입력 컬럼 확정
- 05번: baseline 모델 학습
- 06번: 모델 평가 및 threshold 검토
- 07번: Agent 전달용 산출물 생성

`data/processed/`는 `.gitignore` 대상이므로 GitHub에는 올라가지 않는다.

In [15]:

time_context_profile = pd.DataFrame()
if not ml_window_dataset.empty:
    profile_columns = [column for column in ['manufacturer', 'configuration_type', 'season_bucket', 'split_regime_based'] if column in ml_window_dataset.columns]
    metric_columns = [column for column in ['hour_of_day', 'is_heating_season', 'is_weekend', 'days_since_last_any_event'] if column in ml_window_dataset.columns]
    time_context_profile = (
        ml_window_dataset
        .groupby(profile_columns, dropna=False)[metric_columns]
        .agg(['mean', 'median'])
        .reset_index()
    )
    time_context_profile.columns = ['_'.join([str(part) for part in column if part]).strip('_') for column in time_context_profile.columns.to_flat_index()]

ml_window_dataset.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
if not normal_profile_by_group.empty:
    normal_profile_by_group.to_csv(NORMAL_PROFILE_PATH, index=False, encoding='utf-8-sig')
if not normal_reference_drift_by_group.empty:
    normal_reference_drift_by_group.to_csv(NORMAL_REFERENCE_DRIFT_PATH, index=False, encoding='utf-8-sig')
if not time_context_profile.empty:
    time_context_profile.to_csv(TIME_CONTEXT_PROFILE_PATH, index=False, encoding='utf-8-sig')

print(f'?? ??: {OUTPUT_PATH}')
print(f'?? ??: {NORMAL_PROFILE_PATH}')
print(f'?? ??: {NORMAL_REFERENCE_DRIFT_PATH}')
print(f'?? ??: {TIME_CONTEXT_PROFILE_PATH}')
print(f'?? ? ?: {len(ml_window_dataset):,}')
print(f'?? ? ?: {ml_window_dataset.shape[1]:,}')


?? ??: C:\Project3\HeatGrid_Agent\data\processed\ml_windows\ml_window_dataset.csv
?? ??: C:\Project3\HeatGrid_Agent\data\processed\ml_windows\normal_profile_by_group.csv
?? ??: C:\Project3\HeatGrid_Agent\data\processed\ml_windows\normal_reference_drift_by_group.csv
?? ??: C:\Project3\HeatGrid_Agent\data\processed\ml_windows\window_time_context_profile.csv
?? ? ?: 3,270
?? ? ?: 252


## 9. 다음 단계

03번 결과는 아직 최종 모델 입력이 아니다.

다음 04번에서는 다음을 정리한다.

- 학습에 사용할 feature 컬럼 선택
- 결측률이 높은 feature 제외
- 제조사 공통 feature와 제조사별 feature 분리
- label별 학습 가능성 검토
- Agent 설명에 필요한 센서명과 feature명 정리

03번의 핵심은 `raw 시계열을 Agent가 해석 가능한 ML 학습 단위로 바꾸는 것`이다.